In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from pathlib import Path


In [ ]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ResNet50 standard input size is 224x224 and requires ImageNet normalization
data_transform = transforms.Compose([
    transforms.Resize(size=(224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5), # Tambahan augmentasi vertikal untuk citra sel
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalisasi ImageNet (Krusial)
])

def build_dataloaders(root_dir: str, batch_size: int = 64):
    root = Path(root_dir)
    train_dir = root / 'Train'
    val_dir = root / 'Val'
    
    train_data = datasets.ImageFolder(root=train_dir, transform=data_transform)
    val_data = datasets.ImageFolder(root=val_dir, transform=data_transform)
    
    train_loader = DataLoader(dataset=train_data, batch_size=batch_size, num_workers=0, shuffle=True)
    val_loader = DataLoader(dataset=val_data, batch_size=batch_size, num_workers=0, shuffle=False)
    
    return train_data, val_data, train_loader, val_loader

class ResNet50Model(nn.Module):
    def __init__(self, num_classes):
        super(ResNet50Model, self).__init__()
        # Initialize ResNet50
        self.model = models.resnet50(weights="DEFAULT")
        
        # Replace the final fully connected layer
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, num_classes)
        )
    
    def forward(self, x: torch.Tensor):
        return self.model(x)

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer):
    model.train()
    train_loss, train_acc = 0, 0
    
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        
        y_pred = model(X)
            
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)
        
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module):
    model.eval()
    test_loss, test_acc = 0, 0
    
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            test_pred_logits = model(X)
            
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))
            
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          scheduler: torch.optim.lr_scheduler.LRScheduler, # Tambahan parameter scheduler
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 30,
          checkpoint_path: str = "checkpoint/resnet50/best.pt"):
    
    best_acc = 0
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    
    # Create checkpoint directory if it doesn't exist
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer)
        
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn)
        
        # Step the scheduler
        current_lr = optimizer.param_groups[0]["lr"]
        if scheduler:
            scheduler.step()
        
        print(f"Epoch: {epoch+1} | LR: {current_lr:.6f} | train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | test_loss: {test_loss:.4f} | test_acc: {test_acc:.4f}")
        
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Saving model with test accuracy: {test_acc:.4f} -> {checkpoint_path}")
            
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
        
    return results


In [ ]:
if __name__ == "__main__":
    # List of 10 variants to train
    variants = ["colordis", "deep", "kotera", "mk", "reinhard", "wbc", "xiao", "cagan", "macenko", "vahadane"]
    NUM_EPOCHS = 30
    BATCH_SIZE = 64 # Bisa diadjust sesuai memory GPU
    
    for name in variants:
        print(f"\n{'='*40}")
        print(f"=== Training variant: {name.upper()} ===")
        print(f"{'='*40}")
        
        base_path = Path("data") / name
        if not base_path.exists():
            print(f"Warning: {base_path} not found. Skipping {name}...")
            continue
            
        train_data_v, val_data_v, train_loader_v, val_loader_v = build_dataloaders(base_path, batch_size=BATCH_SIZE)
        
        # Initialize model
        model_v = ResNet50Model(num_classes=len(train_data_v.classes)).to(device)
        
        loss_fn_v = nn.CrossEntropyLoss()
        optimizer_v = torch.optim.Adam(params=model_v.parameters(), lr=0.001, weight_decay=1e-4)
        
        # Tambahan LR Scheduler (Mengurangi LR sebesar 0.1 setiap 10 epoch)
        scheduler_v = torch.optim.lr_scheduler.StepLR(optimizer_v, step_size=10, gamma=0.1)
        
        ckpt_path = f"checkpoint/resnet50/best{name.capitalize()}.pt"
        
        # Start Training
        results_v = train(model=model_v,
                          train_dataloader=train_loader_v,
                          test_dataloader=val_loader_v,
                          optimizer=optimizer_v,
                          scheduler=scheduler_v,
                          loss_fn=loss_fn_v,
                          epochs=NUM_EPOCHS,
                          checkpoint_path=ckpt_path)
        
        print(f"Completed training for variant: {name}")
